# Step 4: Model Training
## Wing Shop Demand Forecasting Project

This notebook trains and evaluates multiple forecasting models.

**Objectives:**
- Train baseline models (Moving Average, Exponential Smoothing)
- Train advanced models (ARIMA, Prophet)
- Create ensemble model
- Evaluate and compare all models
- Target: <10% MAPE (Mean Absolute Percentage Error)

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

# Try to import advanced libraries
try:
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    STATSMODELS_AVAILABLE = True
    print("statsmodels available for ARIMA")
except ImportError:
    STATSMODELS_AVAILABLE = False
    print("Warning: statsmodels not installed. ARIMA models will be unavailable.")

try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
    print("Prophet available")
except ImportError:
    PROPHET_AVAILABLE = False
    print("Warning: prophet not installed. Prophet models will be unavailable.")

In [ ]:
# Setup directories
BASE_DIR = os.path.dirname(os.getcwd()) if 'notebooks' in os.getcwd() else os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, 'data')
MODELS_DIR = os.path.join(BASE_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

print(f"Data Directory: {DATA_DIR}")
print(f"Models Directory: {MODELS_DIR}")

In [ ]:
# Utility functions
def calculate_mape(actual, predicted):
    """Calculate Mean Absolute Percentage Error"""
    actual, predicted = np.array(actual), np.array(predicted)
    mask = actual != 0
    return np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100

def calculate_rmse(actual, predicted):
    """Calculate Root Mean Squared Error"""
    return np.sqrt(np.mean((np.array(actual) - np.array(predicted)) ** 2))

def save_model(model, name):
    """Save model to pickle file"""
    filepath = os.path.join(MODELS_DIR, f"{name}.pkl")
    with open(filepath, 'wb') as f:
        pickle.dump(model, f)
    print(f"Model saved: {filepath}")

def load_model(name):
    """Load model from pickle file"""
    filepath = os.path.join(MODELS_DIR, f"{name}.pkl")
    if os.path.exists(filepath):
        with open(filepath, 'rb') as f:
            return pickle.load(f)
    return None

In [ ]:
# Load data
train_path = os.path.join(DATA_DIR, 'train_data.csv')
test_path = os.path.join(DATA_DIR, 'test_data.csv')

train = pd.read_csv(train_path, parse_dates=['date'])
test = pd.read_csv(test_path, parse_dates=['date'])

print(f"Train data loaded: {train.shape}")
print(f"Test data loaded: {test.shape}")

In [ ]:
# Prepare time series data
def prepare_time_series(df, product=None, store=None):
    """Prepare time series data for a specific product/store"""
    df_filtered = df.copy()
    
    product_col = 'family' if 'family' in df_filtered.columns else 'product'
    if product and product_col in df_filtered.columns:
        df_filtered = df_filtered[df_filtered[product_col] == product]
    
    if store and 'store_nbr' in df_filtered.columns:
        df_filtered = df_filtered[df_filtered['store_nbr'] == store]
    
    daily = df_filtered.groupby('date')['unit_sales'].sum().reset_index()
    daily = daily.sort_values('date')
    daily = daily.set_index('date')
    
    return daily

# Prepare overall time series
train_ts = prepare_time_series(train)
test_ts = prepare_time_series(test)

print(f"Training samples: {len(train_ts)}")
print(f"Testing samples: {len(test_ts)}")

## 4.1 Define Model Classes

In [ ]:
class MovingAverageModel:
    """Simple Moving Average baseline model"""
    
    def __init__(self, window=7):
        self.window = window
        self.history = None
        
    def fit(self, data):
        self.history = data.copy()
        return self
    
    def predict(self, steps=7):
        predictions = []
        history = list(self.history['unit_sales'].values)
        
        for _ in range(steps):
            ma = np.mean(history[-self.window:])
            predictions.append(ma)
            history.append(ma)
        
        return np.array(predictions)
    
    def __str__(self):
        return f"MovingAverage(window={self.window})"


class ExponentialSmoothingModel:
    """Simple Exponential Smoothing model"""
    
    def __init__(self, alpha=0.3):
        self.alpha = alpha
        self.smoothed_value = None
        
    def fit(self, data):
        values = data['unit_sales'].values
        self.smoothed_value = values[0]
        
        for val in values[1:]:
            self.smoothed_value = self.alpha * val + (1 - self.alpha) * self.smoothed_value
        
        return self
    
    def predict(self, steps=7):
        return np.array([self.smoothed_value] * steps)
    
    def __str__(self):
        return f"ExponentialSmoothing(alpha={self.alpha})"


class SeasonalNaiveModel:
    """Seasonal Naive model - uses same day last week"""
    
    def __init__(self, season_length=7):
        self.season_length = season_length
        self.history = None
        
    def fit(self, data):
        self.history = data.copy()
        return self
    
    def predict(self, steps=7):
        predictions = []
        history_values = self.history['unit_sales'].values
        
        for i in range(steps):
            idx = len(history_values) - self.season_length + (i % self.season_length)
            predictions.append(history_values[idx])
        
        return np.array(predictions)
    
    def __str__(self):
        return f"SeasonalNaive(season={self.season_length})"

print("Model classes defined!")

## 4.2 Train Baseline Models

In [ ]:
print("=" * 60)
print("STEP 4: MODEL TRAINING")
print("=" * 60)

print("\n[1] BASELINE MODELS")
print("-" * 40)

results = {}
predictions = {}
models = {}

# Moving Average (7-day)
print("\n  Training Moving Average (window=7)...")
models['ma7'] = MovingAverageModel(window=7)
models['ma7'].fit(train_ts)
predictions['ma7'] = models['ma7'].predict(steps=len(test_ts))
actual = test_ts['unit_sales'].values
results['ma7'] = {
    'mape': calculate_mape(actual, predictions['ma7']),
    'rmse': calculate_rmse(actual, predictions['ma7'])
}
print(f"    MAPE: {results['ma7']['mape']:.2f}%")
print(f"    RMSE: {results['ma7']['rmse']:.2f}")

In [ ]:
# Moving Average (14-day)
print("\n  Training Moving Average (window=14)...")
models['ma14'] = MovingAverageModel(window=14)
models['ma14'].fit(train_ts)
predictions['ma14'] = models['ma14'].predict(steps=len(test_ts))
results['ma14'] = {
    'mape': calculate_mape(actual, predictions['ma14']),
    'rmse': calculate_rmse(actual, predictions['ma14'])
}
print(f"    MAPE: {results['ma14']['mape']:.2f}%")
print(f"    RMSE: {results['ma14']['rmse']:.2f}")

In [ ]:
# Exponential Smoothing
print("\n  Training Exponential Smoothing (alpha=0.3)...")
models['es'] = ExponentialSmoothingModel(alpha=0.3)
models['es'].fit(train_ts)
predictions['es'] = models['es'].predict(steps=len(test_ts))
results['es'] = {
    'mape': calculate_mape(actual, predictions['es']),
    'rmse': calculate_rmse(actual, predictions['es'])
}
print(f"    MAPE: {results['es']['mape']:.2f}%")
print(f"    RMSE: {results['es']['rmse']:.2f}")

In [ ]:
# Seasonal Naive
print("\n  Training Seasonal Naive...")
models['snaive'] = SeasonalNaiveModel(season_length=7)
models['snaive'].fit(train_ts)
predictions['snaive'] = models['snaive'].predict(steps=len(test_ts))
results['snaive'] = {
    'mape': calculate_mape(actual, predictions['snaive']),
    'rmse': calculate_rmse(actual, predictions['snaive'])
}
print(f"    MAPE: {results['snaive']['mape']:.2f}%")
print(f"    RMSE: {results['snaive']['rmse']:.2f}")

## 4.3 Train Advanced Models

In [ ]:
print("\n[2] ADVANCED MODELS")
print("-" * 40)

# ARIMA
if STATSMODELS_AVAILABLE:
    print("\n  Training ARIMA(2,1,2)...")
    try:
        arima_model = ARIMA(train_ts['unit_sales'], order=(2, 1, 2))
        models['arima'] = arima_model.fit()
        predictions['arima'] = models['arima'].forecast(steps=len(test_ts))
        results['arima'] = {
            'mape': calculate_mape(actual, predictions['arima']),
            'rmse': calculate_rmse(actual, predictions['arima'])
        }
        print(f"    MAPE: {results['arima']['mape']:.2f}%")
        print(f"    RMSE: {results['arima']['rmse']:.2f}")
    except Exception as e:
        print(f"    Error: {str(e)}")
        models['arima'] = None
        predictions['arima'] = None
        results['arima'] = None
else:
    print("\n  ARIMA: Skipped (statsmodels not installed)")
    models['arima'] = None
    predictions['arima'] = None
    results['arima'] = None

In [ ]:
# Prophet
if PROPHET_AVAILABLE:
    print("\n  Training Prophet...")
    try:
        prophet_train = train_ts.reset_index()
        prophet_train.columns = ['ds', 'y']
        
        prophet_model = Prophet(
            daily_seasonality=False,
            weekly_seasonality=True,
            yearly_seasonality=True,
            changepoint_prior_scale=0.05
        )
        prophet_model.fit(prophet_train)
        
        future = pd.DataFrame({'ds': test_ts.index})
        forecast = prophet_model.predict(future)
        
        models['prophet'] = prophet_model
        predictions['prophet'] = forecast['yhat'].values
        results['prophet'] = {
            'mape': calculate_mape(actual, predictions['prophet']),
            'rmse': calculate_rmse(actual, predictions['prophet'])
        }
        print(f"    MAPE: {results['prophet']['mape']:.2f}%")
        print(f"    RMSE: {results['prophet']['rmse']:.2f}")
    except Exception as e:
        print(f"    Error: {str(e)}")
        models['prophet'] = None
        predictions['prophet'] = None
        results['prophet'] = None
else:
    print("\n  Prophet: Skipped (prophet not installed)")
    models['prophet'] = None
    predictions['prophet'] = None
    results['prophet'] = None

## 4.4 Create Ensemble Model

In [ ]:
print("\n[3] ENSEMBLE MODEL")
print("-" * 40)

print("\n  Creating Ensemble Model...")

valid_preds = {k: v for k, v in predictions.items() if v is not None}

if len(valid_preds) > 0:
    # Equal weight ensemble
    ensemble_pred = np.zeros(len(test_ts))
    for name, pred in valid_preds.items():
        ensemble_pred += pred / len(valid_preds)
    
    predictions['ensemble'] = ensemble_pred
    results['ensemble'] = {
        'mape': calculate_mape(actual, ensemble_pred),
        'rmse': calculate_rmse(actual, ensemble_pred)
    }
    print(f"    MAPE: {results['ensemble']['mape']:.2f}%")
    print(f"    RMSE: {results['ensemble']['rmse']:.2f}")
else:
    print("    No valid predictions for ensemble")
    predictions['ensemble'] = None
    results['ensemble'] = None

## 4.5 Model Comparison Summary

In [ ]:
print("\n" + "=" * 60)
print("MODEL COMPARISON SUMMARY")
print("=" * 60)
print(f"\n{'Model':<25} {'MAPE (%)':<12} {'RMSE':<12} {'Status'}")
print("-" * 60)

best_mape = float('inf')
best_model = None

for name, metrics in results.items():
    if metrics is not None:
        status = "✓ Target Met" if metrics['mape'] < 10 else "○ Above Target"
        print(f"{name:<25} {metrics['mape']:<12.2f} {metrics['rmse']:<12.2f} {status}")
        
        if metrics['mape'] < best_mape:
            best_mape = metrics['mape']
            best_model = name
    else:
        print(f"{name:<25} {'N/A':<12} {'N/A':<12} ✗ Failed")

print("-" * 60)
print(f"\nBest Model: {best_model} (MAPE: {best_mape:.2f}%)")

target_met = best_mape < 10
if target_met:
    print(f"✓ TARGET ACHIEVED: Error margin < 10%")
else:
    print(f"○ TARGET NOT MET: Need to improve model or data")

In [ ]:
# Visualize predictions vs actual
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(test_ts.index, actual, label='Actual', color='#3b82f6', linewidth=2)

colors = ['#22c55e', '#a855f7', '#f97316', '#eab308', '#ef4444', '#06b6d4']
for i, (name, pred) in enumerate(predictions.items()):
    if pred is not None:
        ax.plot(test_ts.index, pred, label=name, linestyle='--', alpha=0.7, color=colors[i % len(colors)])

ax.set_title('Forecast Comparison: Actual vs Predicted', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Sales (units)')
ax.legend(loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4.6 Save Models

In [ ]:
# Save best model
if best_model and models.get(best_model):
    save_model({
        'model': models[best_model],
        'name': best_model,
        'mape': best_mape,
        'product': 'All Products'
    }, 'best_model_all_products')

# Save all results
save_model({
    'models': models,
    'predictions': predictions,
    'results': results,
    'test_data': test_ts,
    'train_data': train_ts
}, 'all_models')

print("\n" + "=" * 60)
print("MODEL TRAINING COMPLETE!")
print("=" * 60)